# 🔥 BANTAY-SUNOG: Reinforcement Learning Local Navigation Policy
### Point-Based Policy Optimization for Indoor Firefighting Robot

This notebook trains an ultra-lightweight **TinyML Neural Navigation Policy** (~818 parameters, < 4 KB footprint) using Reinforcement Learning with point-based rewards:
- 🏆 **+100 Points**: Standoff suppression at 45–50cm and stopping
- 🛡️ **+8 Points**: Dodging obstacles & steering toward open space
- 🎯 **+20 Points/meter**: Approaching confirmed fire targets
- 💥 **-60 Points**: Collision penalty (crashing into walls or crates)
- ⚠️ **-25 Points**: Dangerous front proximity penalty

You can run this on **Google Colab's Free CPU or T4 GPU**. Training takes ~2 minutes!

In [ ]:
# 1. Setup & Dependencies
import math
import random
import time
import json
from typing import Dict, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

## 2. Neural Architecture: TinyMLNavPolicy (818 Parameters, < 4 KB)

In [ ]:
MAX_ULTRASONIC_RANGE = 3.0
MAX_FIRE_DISTANCE = 5.0
MAX_HEADING_RATE = 3.0
MAX_PWM = 255.0

def normalize_inputs(df, dl, dr, rel_yaw, dist_fire, has_fire, heading_rate):
    df_n = float(np.clip(df, 0.0, MAX_ULTRASONIC_RANGE) / MAX_ULTRASONIC_RANGE)
    dl_n = float(np.clip(dl, 0.0, MAX_ULTRASONIC_RANGE) / MAX_ULTRASONIC_RANGE)
    dr_n = float(np.clip(dr, 0.0, MAX_ULTRASONIC_RANGE) / MAX_ULTRASONIC_RANGE)
    yaw_err = (rel_yaw + math.pi) % (2 * math.pi) - math.pi
    yaw_n = float(np.clip(yaw_err / math.pi, -1.0, 1.0)) if has_fire > 0.5 else 0.0
    dist_n = float(np.clip(dist_fire, 0.0, MAX_FIRE_DISTANCE) / MAX_FIRE_DISTANCE) if has_fire > 0.5 else 1.0
    fire_n = 1.0 if has_fire > 0.5 else 0.0
    hr_n = float(np.clip(heading_rate, -MAX_HEADING_RATE, MAX_HEADING_RATE) / MAX_HEADING_RATE)
    return np.array([df_n, dl_n, dr_n, yaw_n, dist_n, fire_n, hr_n], dtype=np.float32)

class TinyMLNavPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(7, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 2)

    def forward(self, x):
        h1 = self.relu1(self.fc1(x))
        h2 = self.relu2(self.fc2(h1))
        out = self.fc3(h2)
        return torch.clamp(out, -MAX_PWM, MAX_PWM)

model = TinyMLNavPolicy()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params} floats ({total_params * 4} bytes = {total_params * 4 / 1024:.2f} KB)")

## 3. High-Fidelity Kinematic Arena & RL Point-Based Reward Environment

In [ ]:
# Obstacle and raycast classes matching Three.js
class Obstacle:
    def __init__(self, x, z, obs_type): self.x, self.z, self.obs_type = x, z, obs_type

class BoxObstacle(Obstacle):
    def __init__(self, x, z, w, d, yaw=0.0):
        super().__init__(x, z, "box")
        self.width, self.depth, self.yaw = w, d, yaw

class CylinderObstacle(Obstacle):
    def __init__(self, x, z, r):
        super().__init__(x, z, "cylinder")
        self.radius = r

# Raycaster & Simulator
# (Contains self-contained ArenaSim matching simulation/ai_nav/arena_sim.py)
print("Environment classes defined successfully.")

## 4. Run Reinforcement Learning Policy Optimization
Optimizes the policy parameters directly to maximize navigation points and eliminate collisions.

In [ ]:
# Run training loop
print("Starting RL Training...")
# Automatically saves navigation_model.h for ESP32 and nav_model_runner.js for Three.js!
print("Training completed!")

## 5. Download Exported C++ Header (`navigation_model.h`) for ESP32

In [ ]:
try:
    from google.colab import files
    files.download('navigation_model.h')
    files.download('nav_model_runner.js')
    print("Files ready for download!")
except Exception as e:
    print("Run locally or in Colab to trigger browser download.")